<a href="https://colab.research.google.com/github/Abdulkadir03/Crop-Recommender-for-Sahel-Region/blob/main/notebooks/08_rf_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08: Random forest baseline

**Purpose:** Rebuild the original approach (a multi-output random forest) under spatial cross-validation, to see how well it ranks crops and whether it keeps recommending the same few. This is the baseline that LightGBM with constraints (notebook 09) has to beat.

**Inputs:** `training_table.parquet` and `training_config.json` (notebook 07).

**Method:**
1. Features are averaged within each SPAM pixel (21,128 rows, the resolution of the labels), so each pixel counts once.
2. A multi-output `RandomForestRegressor` predicts the 16 crop shares jointly, with the 5 spatial folds from notebook 07 (100 km blocks). Missing values are imputed with the training-fold median.
3. Two feature sets: environment only (climate, soil, terrain, salinity) and all features (adds Sentinel and MODIS).
4. Two naive baselines: the training-fold mean share, and the training-fold mean share per country.
5. Out-of-fold metrics: per-crop rank correlation and R2; overlap between predicted and observed top-5 crops; and diversity of the recommended top-5 sets.

**Outputs** (Google Drive): `oof_rf_env.parquet` and `oof_rf_all.parquet` (out-of-fold predictions per pixel), `results_rf_baseline.csv`.

**Notes:**
- Scores measure agreement with SPAM labels, which are themselves modelled; agreement with field observations is tested in notebook 10.
- To be completed with results.

**Results (out-of-fold, spatial CV, agreement with SPAM):**

| Method | Mean rank correlation | Top-5 overlap | Top-1 match | Distinct top-5 sets |
|---|---|---|---|---|
| Observed SPAM | 1.000 | 1.000 | 1.000 | 430 |
| Baseline: global mean share | -0.054 | 0.733 | 0.311 | 1 |
| Baseline: mean share per country | 0.454 | 0.718 | 0.364 | 12 |
| RF, environment only | 0.644 | 0.789 | 0.591 | 147 |
| RF, all features | 0.639 | 0.790 | 0.592 | 135 |

- Top-5 overlap is a weak yardstick (chance 0.31, a constant guess of the five most common crops scores 0.73); rank correlation and top-1 match separate the methods better.
- Sentinel and MODIS features add nothing, so the environment-only set (51 features) is used from here on.
- Rare crops (sugarcane, tomato, soybean, onion) are poorly predicted by a joint model; beans and cowpea are explained well by country means, which reflects the administrative statistics behind SPAM.
- Compressed model: 53.9 MB; fitting on all pixels: 88 s; predicting 100,000 cells: 3 s.

In [ ]:
# @title Cell 1: load and aggregate to pixels
import os, json, time
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive'
P = f'{BASE}/crop_project'

cfg = json.load(open(f'{P}/training_config.json'))
FEATURES, GROUPS, CROPS = cfg['features'], cfg['groups'], cfg['crops']
SHARE = [f'share_{c}' for c in CROPS]
cols = ['px', 'py', 'fold', 'adm0', 'lon', 'lat', 'has_label'] + FEATURES + SHARE
tab = pd.read_parquet(f'{P}/training_table.parquet', columns=cols)

lab = tab[tab.has_label]
g = lab.groupby(['px', 'py'])
pix = (g[FEATURES].mean().join(g[SHARE].first()).join(g[['fold']].first())
       .join(g[['lon', 'lat']].mean()).join(g['adm0'].agg(lambda s: s.mode().iat[0]))
       .reset_index())
Y = pix[SHARE].to_numpy('float64')
folds = pix.fold.to_numpy()
print('Pixels:', len(pix), '| features:', len(FEATURES), '| crops:', len(CROPS))
print('Pixels per fold:', np.bincount(folds))

In [ ]:
# @title Cell 2: metrics and naive baselines
from scipy.stats import spearmanr
from sklearn.metrics import r2_score

K = 5
def top_mask(M, k=K):
    idx = np.argsort(-M, axis=1, kind='stable')[:, :k]
    m = np.zeros(M.shape, dtype=bool)
    np.put_along_axis(m, idx, True, axis=1)
    return m, idx

def overlap(Pr):
    ob, _ = top_mask(Y); pb, _ = top_mask(Pr)
    return (ob & pb).sum(1) / K

def summarize(Pr, name):
    _, pi = top_mask(Pr); pb, _ = top_mask(Pr)
    sets = pd.Series([tuple(sorted(r)) for r in pi]).value_counts()
    rho = np.nanmean([spearmanr(Y[:, j], Pr[:, j]).correlation for j in range(Y.shape[1])])
    return {'method': name, 'mean_spearman': round(float(rho), 3),
            'top5_overlap': round(float(overlap(Pr).mean()), 3),
            'top1_match': round(float((np.argmax(Pr, 1) == np.argmax(Y, 1)).mean()), 3),
            'distinct_top5_sets': len(sets),
            'most_common_set_share': round(float(sets.iloc[0] / len(Pr)), 3),
            'crops_in_top5_over_1pct': int((pb.mean(0) >= 0.01).sum())}

def baseline_global():
    Pr = np.zeros_like(Y)
    for k in range(5):
        Pr[folds == k] = Y[folds != k].mean(0)
    return Pr

def baseline_country():
    Pr = np.zeros_like(Y)
    for k in range(5):
        tr, te = folds != k, folds == k
        gm = Y[tr].mean(0)
        means = pd.DataFrame(Y[tr]).groupby(pix.adm0.to_numpy()[tr]).mean()
        cm = means.reindex(pix.adm0.to_numpy()[te])
        Pr[te] = np.where(cm.isna().to_numpy(), gm, cm.to_numpy())
    return Pr

Pg, Pc = baseline_global(), baseline_country()
print('Baselines ready')

In [ ]:
# @title Cell 3: random forest with spatial cross-validation
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

N_TREES, SEED = 150, 42
ENV = GROUPS['climate'] + GROUPS['soil'] + GROUPS['terrain'] + GROUPS['salinity']

def make_rf():
    return RandomForestRegressor(n_estimators=N_TREES, min_samples_leaf=5, max_features=0.33,
                                 max_samples=0.7, n_jobs=-1, random_state=SEED)

def cv_rf(feats, tag):
    Pr = np.zeros_like(Y)
    X = pix[feats]
    for k in range(5):
        t0 = time.time()
        tr, te = folds != k, folds == k
        imp = SimpleImputer(strategy='median').fit(X[tr])
        rf = make_rf().fit(imp.transform(X[tr]), Y[tr])
        Pr[te] = rf.predict(imp.transform(X[te]))
        print(f'{tag}: fold {k} done in {time.time() - t0:.0f} s')
    out = pix[['px', 'py', 'fold']].copy()
    out[CROPS] = Pr
    out.to_parquet(f'{P}/oof_rf_{tag}.parquet', index=False)
    return Pr

P_env = cv_rf(ENV, 'env')
P_all = cv_rf(FEATURES, 'all')

In [ ]:
# @title Cell 4: results
rows = [summarize(Y, 'observed SPAM (reference)'), summarize(Pg, 'baseline: mean share'),
        summarize(Pc, 'baseline: mean share per country'),
        summarize(P_env, 'RF: environment only'), summarize(P_all, 'RF: all features')]
res = pd.DataFrame(rows).set_index('method')
pd.set_option('display.width', 220)
print(res.to_string())

def rho(Pr, j):
    return spearmanr(Y[:, j], Pr[:, j]).correlation
pc = pd.DataFrame({'mean_share': Y.mean(0),
                   'rho_country_base': [rho(Pc, j) for j in range(len(CROPS))],
                   'rho_env': [rho(P_env, j) for j in range(len(CROPS))],
                   'rho_all': [rho(P_all, j) for j in range(len(CROPS))],
                   'r2_env': [r2_score(Y[:, j], P_env[:, j]) for j in range(len(CROPS))],
                   'r2_all': [r2_score(Y[:, j], P_all[:, j]) for j in range(len(CROPS))]},
                  index=CROPS).round(3)
print('\nPer crop:'); print(pc)

bycty = pd.DataFrame({'pixels': pd.Series(1, index=range(len(pix))).groupby(pix.adm0.to_numpy()).sum(),
                      'country_base': pd.Series(overlap(Pc)).groupby(pix.adm0.to_numpy()).mean(),
                      'rf_env': pd.Series(overlap(P_env)).groupby(pix.adm0.to_numpy()).mean(),
                      'rf_all': pd.Series(overlap(P_all)).groupby(pix.adm0.to_numpy()).mean()}).round(3)
print('\nTop-5 overlap by country:'); print(bycty)
byfold = pd.DataFrame({'country_base': pd.Series(overlap(Pc)).groupby(folds).mean(),
                       'rf_env': pd.Series(overlap(P_env)).groupby(folds).mean(),
                       'rf_all': pd.Series(overlap(P_all)).groupby(folds).mean()}).round(3)
print('\nTop-5 overlap by fold:'); print(byfold)
res.to_csv(f'{P}/results_rf_baseline.csv')

In [ ]:
# @title Cell 5: model size and prediction time
import joblib
imp = SimpleImputer(strategy='median').fit(pix[FEATURES])
t0 = time.time(); rf = make_rf().fit(imp.transform(pix[FEATURES]), Y)
print(f'Final fit on all pixels: {time.time() - t0:.0f} s')
joblib.dump(rf, '/content/rf_tmp.joblib', compress=3)
print('Compressed model size (MB):', round(os.path.getsize('/content/rf_tmp.joblib') / 1e6, 1))
Xc = imp.transform(tab[tab.has_label][FEATURES].sample(100000, random_state=0))
t0 = time.time(); rf.predict(Xc); print(f'Predicting 100,000 cells: {time.time() - t0:.0f} s')
os.remove('/content/rf_tmp.joblib')
drive.flush_and_unmount(); drive.mount('/content/drive')

In [ ]:
# @title Cell 6: which crops never reach the top five?
ob, _ = top_mask(Y)
freq = pd.DataFrame({'observed': ob.mean(0),
                     'RF env': top_mask(P_env)[0].mean(0),
                     'RF all': top_mask(P_all)[0].mean(0)}, index=CROPS).round(3)
print('Share of pixels where each crop is in the top 5:')
print(freq)

In [ ]:
for tag in ['env', 'all']:
    o = pd.read_parquet(f'{P}/oof_rf_{tag}.parquet')
    arr = pix[['px', 'py']].merge(o, on=['px', 'py'], how='left')[CROPS].to_numpy()
    if tag == 'env':
        P_env = arr
    else:
        P_all = arr
print(P_env.shape, P_all.shape, 'missing values:', int(np.isnan(P_env).sum()), int(np.isnan(P_all).sum()))